In [2]:
import pandas as pd
import numpy as np

# Load raw data
activity = pd.read_csv("../data/FitbitActivity.csv")
sleep = pd.read_csv("../data/FitbitSleep.csv")
survey = pd.read_csv("../data/BasicSurvey.csv")

C:\Users\yodit\AppData\Local\Temp\ipykernel_26456\260619536.py:7: DtypeWarning: Columns (0: prescriptiondrugabuse3rc_1, 1: mjeset3_1, 2: mjesubset3_1, 3: recreationaldrugs3rc_2, 4: prescriptiondrugabuse1rc_2, 5: prescriptiondrugabuse2rc_2, 6: mjeset2_3, 7: mjesubset2_3, 8: recreationaldrugs2rc_3, 9: recreationaldrugs3rc_3, 10: prescriptiondrugabuse2rc_3, 11: prescriptiondrugabuse3rc_3, 12: mjeset3_4, 13: mjesubset3_4, 14: disability3rca_4, 15: disability3rcb_4, 16: disability4rca_4, 17: disability4rcb_4, 18: disability5rca_4, 19: disability5rcb_4, 20: sleepdisorderrc_4, 21: recreationaldrugs2rc_4, 22: prescriptiondrugabuse1rc_4, 23: studyAbroad_5_5, 24: studyAbroadF_Other_5, 25: mjeset3_5, 26: mjesubset3_5, 27: vegan_5, 28: disability3rca_5, 29: disability3rcb_5, 30: undiagall3rca_5, 31: undiagall3rcb_5, 32: recreationaldrugs3rc_5, 33: prescriptiondrugabuse1rc_5, 34: prescriptiondrugabuse2rc_5, 35: prescriptiondrugabuse3rc_5, 36: studyAbroadF_5_6, 37: mjeset3_6, 38: mjesubset3_6, 39: v

In [3]:
# Standardize column names
activity.columns = activity.columns.str.lower()
sleep.columns = sleep.columns.str.lower()
survey.columns = survey.columns.str.lower()

In [4]:
# Convert date columns to datetime
activity["datadate"] = pd.to_datetime(activity["datadate"])
sleep["datadate"] = pd.to_datetime(sleep["datadate"])

# Convert survey date columns to datetime
for suffix in ['4', '5', '6', '8']:
    col = f"startdate_{suffix}"
    survey[col] = pd.to_datetime(survey[col], format="%d%b%Y %H:%M:%S", errors="coerce")

In [5]:
# Participants in all three datasets (activity, sleep, survey)
survey_ids = set(survey["egoid"])
activity_ids = set(activity["egoid"])
sleep_ids = set(sleep["egoid"])

# intersection of all three
common_ids = survey_ids & activity_ids & sleep_ids

print("Participants in all three datasets:", len(common_ids))

Participants in all three datasets: 622


In [6]:
# Keep only needed columns
survey_subset = survey[[
    "egoid",
    "stress_4", "startdate_4",
    "stress_5", "startdate_5",
    "stress_6", "startdate_6",
    "stress_8", "startdate_8"
]].copy()

# survey_subset.head(10)

In [7]:
activity.columns

Index(['egoid', 'datadate', 'complypercent', 'meanrate', 'sdrate', 'steps',
       'floors', 'sedentaryminutes', 'lightlyactiveminutes',
       'fairlyactiveminutes', 'veryactiveminutes', 'lowrangemins',
       'fatburnmins', 'cardiomins', 'peakmins', 'lowrangecal', 'fatburncal',
       'cardiocal', 'peakcal'],
      dtype='str')

In [8]:
survey_subset.shape

(722, 9)

In [9]:
# Reshape manually by stacking waves

wave4 = survey_subset[["egoid", "stress_4", "startdate_4"]].copy()
wave4.columns = ["egoid", "stress", "survey_date"]
wave4["wave"] = 4

wave5 = survey_subset[["egoid", "stress_5", "startdate_5"]].copy()
wave5.columns = ["egoid", "stress", "survey_date"]
wave5["wave"] = 5

wave6 = survey_subset[["egoid", "stress_6", "startdate_6"]].copy()
wave6.columns = ["egoid", "stress", "survey_date"]
wave6["wave"] = 6

wave8 = survey_subset[["egoid", "stress_8", "startdate_8"]].copy()
wave8.columns = ["egoid", "stress", "survey_date"]
wave8["wave"] = 8

# Combine all waves
survey_long = pd.concat([wave4, wave5, wave6, wave8], ignore_index=True)

# Remove rows without stress score
survey_long = survey_long.dropna(subset=["stress"])

survey_long.head()

,egoid,stress,survey_date,wave
0,44869,1.9,2016-11-06 10:16:10,4
6,34591,2.1,2016-11-02 14:14:55,4
8,92552,1.7,2016-11-07 17:40:36,4
9,73158,0.9,2016-11-02 16:37:36,4
10,45963,2.8,2016-11-01 14:08:30,4


In [10]:
survey_long.shape

(1245, 4)

In [11]:
# All the rows with 44869 egoid
survey_long[survey_long["egoid"] == 44869]

,egoid,stress,survey_date,wave
0,44869,1.9,2016-11-06 10:16:10,4
722,44869,1.8,2017-04-19 11:33:51,5
1444,44869,2.5,2017-11-13 11:38:42,6


In [12]:
# How many participants appear more than once in survey long
(survey_long['egoid'].value_counts() > 1).sum()

np.int64(373)

In [13]:
# Reapeated surveys
survey_long['egoid'].value_counts().head()

egoid
92552    4
73158    4
45963    4
22138    4
77941    4
Name: count, dtype: int64

In [14]:
len(set(activity.egoid) & set(sleep.egoid) & set(survey_long.egoid))

435

In [15]:
# Filter survey_long to participants present in all three datasets
valid_ids = set(activity.egoid) & set(sleep.egoid) & set(survey_long.egoid)
survey_valid = survey_long[survey_long.egoid.isin(valid_ids)]
survey_valid.shape


(1222, 4)

In [16]:
survey_valid.head()

,egoid,stress,survey_date,wave
0,44869,1.9,2016-11-06 10:16:10,4
6,34591,2.1,2016-11-02 14:14:55,4
8,92552,1.7,2016-11-07 17:40:36,4
9,73158,0.9,2016-11-02 16:37:36,4
10,45963,2.8,2016-11-01 14:08:30,4


In [17]:
# How frequently were surveys taken
survey_valid = survey_valid.copy()
survey_valid['survey_date'] = pd.to_datetime(survey_valid['survey_date'])
survey_valid.sort_values(['egoid','survey_date']).groupby('egoid')['survey_date'].diff().describe()

count                         787
mean     269 days 10:30:40.499364
std      151 days 22:24:24.707426
min             147 days 19:44:20
25%             160 days 23:35:08
50%             212 days 22:41:24
75%      233 days 01:55:29.500000
max             750 days 07:15:29
Name: survey_date, dtype: object

In [18]:
# Remove duplicates, keeping the row with fewer NaNs
activity['_nan_count'] = activity.isna().sum(axis=1)
activity = (
    activity
    .sort_values(by='_nan_count')
    .drop_duplicates(subset=['egoid','datadate'], keep='first')
    .drop(columns=['_nan_count'])
)

In [19]:
activity.groupby(['egoid','datadate']).size().max()

np.int64(1)

In [20]:
sleep = sleep.rename(columns={'dataDate': 'datadate'})

In [21]:
sleep.groupby(['egoid','datadate']).size().max()

np.int64(10)

In [22]:
# Aggregate multiple sleep sessions per day into one daily summary per participant,
# using weighted average for Efficiency (weighted by minutes asleep)

# First create weighted efficiency numerator
sleep['eff_weighted'] = sleep['efficiency'] * sleep['minsasleep']

sleep_daily = (
    sleep
    .groupby(['egoid','datadate'], as_index=False)
    .agg({
        'bedtimedur': 'sum',
        'minstofallasleep': 'sum',
        'minsafterwakeup': 'sum',
        'minsasleep': 'sum',
        'minsawake': 'sum',
        'eff_weighted': 'sum'
    })
)

# Compute proper daily efficiency
sleep_daily['efficiency'] = (
    sleep_daily['eff_weighted'] / sleep_daily['minsasleep']
)

# Drop helper column
sleep_daily = sleep_daily.drop(columns='eff_weighted')

In [23]:
sleep_daily.groupby(['egoid','datadate']).size().max()

np.int64(1)

In [24]:
# Merge activity and sleep
wearable_daily = activity.merge(
    sleep_daily,
    on=['egoid', 'datadate'],
    how='inner'   # keep only days with both
)

In [25]:
wearable_daily.shape

(294930, 25)

In [26]:
wearable_daily.groupby(['egoid','datadate']).size().max()

np.int64(1)

In [27]:
# Sort merged wearable data by participant and date
wearable_daily = wearable_daily.sort_values(
    ['egoid', 'datadate']
)

In [28]:
wearable_daily.head()

,egoid,datadate,complypercent,meanrate,sdrate,steps,floors,sedentaryminutes,lightlyactiveminutes,fairlyactiveminutes,...,lowrangecal,fatburncal,cardiocal,peakcal,bedtimedur,minstofallasleep,minsafterwakeup,minsasleep,minsawake,efficiency
178904,10237,2015-08-04,96,72.825836,22.460846,8112.0,9.0,516.0,274.0,81.0,...,2144.9983,1201.33940,86.561569,0.0,477,5,0,426,46,0.902574
178905,10237,2015-08-05,97,73.927544,20.441671,12478.0,10.0,473.0,359.0,81.0,...,2610.0005,926.24640,96.583168,0.0,504,0,1,463,40,0.920477
178906,10237,2015-08-06,93,74.651184,25.053057,13343.0,11.0,522.0,221.0,65.0,...,1994.4237,1149.47750,474.898560,0.0,498,4,2,454,38,0.922764
178907,10237,2015-08-07,96,68.445740,18.437967,6510.0,1.0,443.0,263.0,36.0,...,2211.6418,699.25714,9.395250,0.0,641,3,0,594,44,0.931601
178908,10237,2015-08-08,98,75.968529,20.436409,9154.0,5.0,607.0,313.0,19.0,...,2478.4670,739.46881,186.276490,0.0,523,2,5,492,24,0.953488


In [29]:
wearable_daily.columns

Index(['egoid', 'datadate', 'complypercent', 'meanrate', 'sdrate', 'steps',
       'floors', 'sedentaryminutes', 'lightlyactiveminutes',
       'fairlyactiveminutes', 'veryactiveminutes', 'lowrangemins',
       'fatburnmins', 'cardiomins', 'peakmins', 'lowrangecal', 'fatburncal',
       'cardiocal', 'peakcal', 'bedtimedur', 'minstofallasleep',
       'minsafterwakeup', 'minsasleep', 'minsawake', 'efficiency'],
      dtype='str')

In [30]:
# Merge wearable data with survey responses and keep only the wearable
# records that fall within the 30-day window before each survey date.
# This links each survey's stress score to the participant's daily
# activity and sleep data from the preceding month.

merged = wearable_daily.merge(
    survey_valid,
    on="egoid",
    how="inner"
)

merged = merged[
    (merged["datadate"] >= merged["survey_date"] - pd.Timedelta(days=30)) &
    (merged["datadate"] <= merged["survey_date"])
]

In [31]:
# Confirm multiple daily rows per student per wave
rows_per_wave = merged.groupby(['egoid', 'wave']).size()
print("Rows per (student, wave):")
print(rows_per_wave.describe())
print(f"\nAll have >1 row: {(rows_per_wave > 1).all()}")
print(f"Min: {rows_per_wave.min()}, Max: {rows_per_wave.max()}")
print(f"Unique (student, wave) groups: {len(rows_per_wave)}")

Rows per (student, wave):
count    993.000000
mean      22.787513
std        8.294732
min        1.000000
25%       19.000000
50%       26.000000
75%       29.000000
max       30.000000
dtype: float64

All have >1 row: False
Min: 1, Max: 30
Unique (student, wave) groups: 993


In [32]:
merged.head()

,egoid,datadate,complypercent,meanrate,sdrate,steps,floors,sedentaryminutes,lightlyactiveminutes,fairlyactiveminutes,...,peakcal,bedtimedur,minstofallasleep,minsafterwakeup,minsasleep,minsawake,efficiency,stress,survey_date,wave
1380,10237,2017-03-12,28,67.382355,5.253840,6573.0,3.0,763.0,194.0,0.0,...,0.0,542,3,0,514,25,0.953618,2.0,2017-04-10 20:36:46,5
1383,10237,2017-03-13,73,78.500000,19.802874,10862.0,6.0,489.0,258.0,58.0,...,0.0,533,2,0,518,13,0.975518,2.0,2017-04-10 20:36:46,5
1386,10237,2017-03-14,91,85.113686,18.501575,13173.0,6.0,425.0,301.0,101.0,...,0.0,597,0,0,573,24,0.959799,2.0,2017-04-10 20:36:46,5
1389,10237,2017-03-15,98,78.355652,15.806064,9987.0,7.0,516.0,282.0,38.0,...,0.0,630,6,2,560,62,0.900322,2.0,2017-04-10 20:36:46,5
1392,10237,2017-03-16,98,83.551430,18.608688,14838.0,6.0,518.0,255.0,132.0,...,0.0,481,0,0,466,15,0.968815,2.0,2017-04-10 20:36:46,5


In [33]:
merged.columns

Index(['egoid', 'datadate', 'complypercent', 'meanrate', 'sdrate', 'steps',
       'floors', 'sedentaryminutes', 'lightlyactiveminutes',
       'fairlyactiveminutes', 'veryactiveminutes', 'lowrangemins',
       'fatburnmins', 'cardiomins', 'peakmins', 'lowrangecal', 'fatburncal',
       'cardiocal', 'peakcal', 'bedtimedur', 'minstofallasleep',
       'minsafterwakeup', 'minsasleep', 'minsawake', 'efficiency', 'stress',
       'survey_date', 'wave'],
      dtype='str')

In [34]:
# Aggregate daily wearable features into per-student, per-wave averages,
# producing one summary row per (student, wave, stress) combination
agg_data = (
    merged
    .groupby(["egoid", "wave", "stress"])
    .mean(numeric_only=True)
    .reset_index()
)

In [35]:
agg_data.shape
agg_data.head()

,egoid,wave,stress,complypercent,meanrate,sdrate,steps,floors,sedentaryminutes,lightlyactiveminutes,...,lowrangecal,fatburncal,cardiocal,peakcal,bedtimedur,minstofallasleep,minsafterwakeup,minsasleep,minsawake,efficiency
0,10237,5,2.0,86.416667,77.725359,18.185866,13729.750000,13.916667,571.083333,236.875000,...,1852.499676,1361.212831,44.861541,25.350632,501.250000,2.666667,0.291667,472.291667,26.000000,0.947869
1,10237,6,2.0,93.133333,78.689774,18.848640,15198.966667,11.000000,596.600000,252.533333,...,1916.359660,1315.487469,133.005992,37.065208,458.066667,2.033333,0.900000,443.566667,11.566667,0.975156
2,10469,4,2.6,86.333333,87.609175,13.235560,7755.533333,7.533333,703.400000,284.533333,...,1130.118397,570.076205,9.065400,7.656000,441.466667,4.000000,1.666667,416.800000,19.000000,0.956601
3,10469,5,2.1,84.437500,88.637704,11.991933,8674.500000,8.562500,776.500000,247.750000,...,1071.946276,575.211165,10.626682,0.579539,395.937500,3.312500,0.437500,361.187500,31.000000,0.924312
4,11002,4,1.1,90.666667,75.031066,16.005319,11169.444444,11.555556,681.666667,184.444444,...,1814.190303,1021.511693,29.530839,11.463622,479.000000,2.444444,0.888889,451.111111,24.555556,0.946898


In [36]:
# Count the number of daily records per student per wave
day_counts = (
    merged
    .groupby(["egoid", "wave"])
    .size()
    .reset_index(name="n_days")
)


In [37]:
# Merge day counts into agg_data, then keep only groups with ≥7 days
agg_data = agg_data.merge(day_counts, on=["egoid", "wave"], how="left")
agg_data = agg_data[agg_data["n_days"] >= 7]
print(f"After filtering: {agg_data.shape}")

After filtering: (912, 27)


In [38]:
agg_data.head()

,egoid,wave,stress,complypercent,meanrate,sdrate,steps,floors,sedentaryminutes,lightlyactiveminutes,...,fatburncal,cardiocal,peakcal,bedtimedur,minstofallasleep,minsafterwakeup,minsasleep,minsawake,efficiency,n_days
0,10237,5,2.0,86.416667,77.725359,18.185866,13729.750000,13.916667,571.083333,236.875000,...,1361.212831,44.861541,25.350632,501.250000,2.666667,0.291667,472.291667,26.000000,0.947869,24
1,10237,6,2.0,93.133333,78.689774,18.848640,15198.966667,11.000000,596.600000,252.533333,...,1315.487469,133.005992,37.065208,458.066667,2.033333,0.900000,443.566667,11.566667,0.975156,30
2,10469,4,2.6,86.333333,87.609175,13.235560,7755.533333,7.533333,703.400000,284.533333,...,570.076205,9.065400,7.656000,441.466667,4.000000,1.666667,416.800000,19.000000,0.956601,15
3,10469,5,2.1,84.437500,88.637704,11.991933,8674.500000,8.562500,776.500000,247.750000,...,575.211165,10.626682,0.579539,395.937500,3.312500,0.437500,361.187500,31.000000,0.924312,16
4,11002,4,1.1,90.666667,75.031066,16.005319,11169.444444,11.555556,681.666667,184.444444,...,1021.511693,29.530839,11.463622,479.000000,2.444444,0.888889,451.111111,24.555556,0.946898,9


In [39]:
# Check how many student-wave groups have very low day counts
print("Groups with 1 day: ", (day_counts["n_days"] == 1).sum())
print("Groups with 2 days:", (day_counts["n_days"] == 2).sum())
print("Groups with 3 days:", (day_counts["n_days"] == 3).sum())
print("Groups with 1–3 days (total):", (day_counts["n_days"] <= 3).sum())
print(f"  out of {len(day_counts)} total groups ({(day_counts['n_days'] <= 3).mean():.1%})")

Groups with 1 day:  16
Groups with 2 days: 18
Groups with 3 days: 12
Groups with 1–3 days (total): 46
  out of 993 total groups (4.6%)


In [40]:
# Drop student-wave groups with fewer than 7 days of wearable data
agg_data = agg_data[agg_data["n_days"] >= 7]

In [41]:
print("Groups with less than 7 days (total):", (agg_data["n_days"] < 7).sum())

Groups with less than 7 days (total): 0


In [42]:
# Group-aware train/test split (80/20): all waves of the same student
# stay in the same split to prevent data leakage across participants
from sklearn.model_selection import GroupShuffleSplit

X = agg_data.drop(columns=["stress", "egoid"])
y = agg_data["stress"]
groups = agg_data["egoid"]

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [43]:
# Verify no student appears in both train and test sets (should return empty set)
set(agg_data.iloc[train_idx]["egoid"]) & set(agg_data.iloc[test_idx]["egoid"])

set()

In [ ]:
# Standardize features to zero mean and unit variance.
# This ensures no feature dominates the linear model simply due to its
# scale (e.g., steps in thousands vs. efficiency in 0–100). It also
# improves numerical stability and makes coefficients directly comparable.
# fit_transform on train only; transform test using train statistics
# to avoid data leakage.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [45]:
# Train a linear regression baseline model on scaled wearable features
# and evaluate on the held-out test set using Mean Absolute Error (MAE)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)

MAE: 0.5829494730489577


In [46]:
# Naive baseline: always predicts the training-set mean stress score.
# If the linear regression MAE is not lower than this, the model
# is not learning anything beyond the average.
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train_scaled, y_train)

y_dummy = dummy.predict(X_test_scaled)

mae_dummy = mean_absolute_error(y_test, y_dummy)
print("Dummy MAE:", mae_dummy)

Dummy MAE: 0.5653779261435068


In [47]:
# Train a Random Forest regressor (nonlinear model) as a comparison
# to the linear baseline. Tree-based models don't need scaled features.
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
print("Random Forest MAE:", mae_rf)

Random Forest MAE: 0.5687356258405173


In [48]:
X_train.shape

(738, 25)

In [49]:
# Aggregate daily wearable features into mean and std per (student, wave, stress),
# capturing both central tendency and day-to-day variability.
# Flatten multi-level columns (e.g. ('steps','mean')) into 'steps_mean', 'steps_std'.
agg_data = (
    merged
    .groupby(["egoid", "wave", "stress"])
    .agg(["mean", "std"])
)

agg_data.columns = ['_'.join(col).strip() for col in agg_data.columns.values]
agg_data = agg_data.reset_index()

In [50]:
# Merge day counts into agg_data, then keep only groups with ≥7 days
agg_data = agg_data.merge(day_counts, on=["egoid", "wave"], how="left")
agg_data = agg_data[agg_data["n_days"] >= 7]
print(f"After filtering: {agg_data.shape}")

After filtering: (912, 54)


In [51]:
agg_data.head()

,egoid,wave,stress,datadate_mean,datadate_std,complypercent_mean,complypercent_std,meanrate_mean,meanrate_std,sdrate_mean,...,minsafterwakeup_std,minsasleep_mean,minsasleep_std,minsawake_mean,minsawake_std,efficiency_mean,efficiency_std,survey_date_mean,survey_date_std,n_days
0,10237,5,2.0,2017-03-27 13:00:00,9 days 04:38:35.981370,86.416667,21.400765,77.725359,5.284242,18.185866,...,0.907896,472.291667,73.144573,26.000000,19.888821,0.947869,0.039987,2017-04-10 20:36:46,0 days,24
1,10237,6,2.0,2017-10-31 12:00:00,8 days 19:16:54.488423,93.133333,11.230296,78.689774,6.113357,18.848640,...,1.470398,443.566667,86.966376,11.566667,7.089007,0.975156,0.014042,2017-11-15 21:19:29,0 days,30
2,10469,4,2.6,2016-10-26 14:24:00,9 days 01:51:31.211221,86.333333,16.872068,87.609175,4.389913,13.235560,...,2.894987,416.800000,133.648900,19.000000,12.235779,0.956601,0.024360,2016-11-13 23:13:43,0 days,15
3,10469,5,2.1,2017-04-09 21:00:00,7 days 17:02:57.057545,84.437500,23.209104,88.637704,3.381754,11.991933,...,1.504161,361.187500,179.697234,31.000000,18.843213,0.924312,0.032662,2017-04-23 22:40:03,0 days,16
4,11002,4,1.1,2016-10-14 10:40:00,3 days 04:56:29.530488,90.666667,17.457090,75.031066,8.311891,16.005319,...,1.615893,451.111111,126.349163,24.555556,9.850268,0.946898,0.019278,2016-11-08 13:32:27,0 days,9


In [52]:
print("Groups with less than 7 days (total):", (agg_data["n_days"] < 7).sum())

Groups with less than 7 days (total): 0


In [53]:
# Group-aware train/test split (80/20): all waves of the same student
# stay in the same split to prevent data leakage across participants
from sklearn.model_selection import GroupShuffleSplit

# Drop non-numeric and non-feature columns
drop_cols = ["stress", "egoid"]
drop_cols += [c for c in agg_data.columns if agg_data[c].dtype.kind in ('M', 'm')]  # datetime/timedelta
X = agg_data.drop(columns=drop_cols)
y = agg_data["stress"]
groups = agg_data["egoid"]

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [54]:
# Verify no student appears in both train and test sets (should return empty set)
set(agg_data.iloc[train_idx]["egoid"]) & set(agg_data.iloc[test_idx]["egoid"])

set()

In [55]:
# Train a Random Forest regressor (nonlinear model) as a comparison
# to the linear baseline. Tree-based models don't need scaled features.
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
print("Random Forest MAE:", mae_rf)

Random Forest MAE: 0.571015638857184


In [56]:
agg_data["stress"].describe()

count    912.000000
mean       1.670115
std        0.636237
min        0.000000
25%        1.200000
50%        1.700000
75%        2.100000
max        4.000000
Name: stress, dtype: float64

In [57]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred_rf)
print("R2:", r2)

R2: -0.026569473745235994


In [58]:
!pip install xgboost


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Train an XGBoost regressor on the 30-day aggregated wearable features.
# XGBoost is a gradient-boosted tree model that often outperforms Random Forest
# by sequentially correcting prediction errors. Evaluate with MAE and R².
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost MAE:", mae_xgb)
print("XGBoost R2:", r2_xgb)

XGBoost MAE: 0.5873541869131925
XGBoost R2: -0.08395018375916474


In [60]:
sleep.columns

Index(['egoid', 'datadate', 'timetobed', 'timeoutofbed', 'bedtimedur',
       'minstofallasleep', 'minsafterwakeup', 'minsasleep', 'minsawake',
       'efficiency', 'eff_weighted'],
      dtype='str')

In [61]:
# Merge wearable data with survey responses and keep only the 7 days
# immediately before each survey date (excluding the survey day itself).
# This creates a short-term activity/sleep window linked to each stress score.
merged_7d = wearable_daily.merge(
    survey_long,
    on="egoid",
    how="inner"
)

merged_7d = merged_7d[
    (merged_7d["datadate"] >= merged_7d["survey_date"] - pd.Timedelta(days=7)) &
    (merged_7d["datadate"] < merged_7d["survey_date"])
]

In [62]:
merged_7d.shape

(5348, 28)

In [63]:
# Aggregate the 7-day pre-survey wearable data into mean and std
# per (student, wave, stress, survey_date), using only numeric columns
# to avoid errors with datetime fields. Flatten column names for modeling.

# Select numeric wearable columns only
numeric_cols = wearable_daily.select_dtypes(include=np.number).columns

agg_7d = merged_7d.groupby(
    ["egoid", "wave", "stress", "survey_date"]
)[numeric_cols].agg(["mean", "std"])

# Flatten multi-index columns
agg_7d.columns = [f"{col}_{stat}" for col, stat in agg_7d.columns]

agg_7d = agg_7d.reset_index()

agg_7d.shape

(899, 52)

In [64]:
# Separate features (X), target (y), and group labels for the 7-day window data.
# Drop non-feature columns; keep egoid separately for group-aware splitting.

# Drop identifiers
X_7d = agg_7d.drop(columns=["egoid", "wave", "stress", "survey_date"])
y_7d = agg_7d["stress"]
groups_7d = agg_7d["egoid"]

In [65]:
# Group-aware train/test split (80/20) for the 7-day window data.
# All waves of the same student stay in one split to prevent data leakage.

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(X_7d, y_7d, groups_7d):
    X_train_7d, X_test_7d = X_7d.iloc[train_idx], X_7d.iloc[test_idx]
    y_train_7d, y_test_7d = y_7d.iloc[train_idx], y_7d.iloc[test_idx]

In [66]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_7d_scaled = scaler.fit_transform(X_train_7d)
X_test_7d_scaled = scaler.transform(X_test_7d)

In [67]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train_7d_scaled, y_train_7d)

y_dummy_7d = dummy.predict(X_test_7d_scaled)

print("Dummy MAE (7d):", mean_absolute_error(y_test_7d, y_dummy_7d))

Dummy MAE (7d): 0.4982583028780693


In [68]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_7d, y_train_7d)

y_pred_rf_7d = rf.predict(X_test_7d)

print("RF MAE (7d):", mean_absolute_error(y_test_7d, y_pred_rf_7d))
print("RF R2 (7d):", r2_score(y_test_7d, y_pred_rf_7d))

RF MAE (7d): 0.49130131377683617
RF R2 (7d): -0.011217413305236512


In [69]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    random_state=42,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4
)

xgb.fit(X_train_7d, y_train_7d)

y_pred_xgb_7d = xgb.predict(X_test_7d)

print("XGBoost MAE (7d):", mean_absolute_error(y_test_7d, y_pred_xgb_7d))
print("XGBoost R2 (7d):", r2_score(y_test_7d, y_pred_xgb_7d))

XGBoost MAE (7d): 0.5091624712616656
XGBoost R2 (7d): -0.09189722017557123


In [70]:
agg_data.to_csv("../data/wearable_features_30d.csv", index=False)
agg_7d.to_csv("../data/wearable_features_7d.csv", index=False)